# <center>Modeling
The previous analysis treated G3 as a continuous numerical outcome and therefore addressed the problem as regression. The classification analysis instead converts the final grade into a binary Pass/Fail outcome. This allows the analysis to address whether student characteristics and academic performance up to the second period (G1 and G2) can be used to predict whether a student ultimately passes.

## Outline:

1. Modeling objectives:
- Classification Task =   
    - **Can we classify students according to their final academic performance?**  
    - **Can we predict whether a student will pass or fail?** 


2. Classification
    - 3.1 Classification objective
    - 3.2 Creating the Pass/Fail target
    - 3.3 Class distribution

    - 3.4 Feature/target separation
    - 3.5 Train/test split
    - 3.6 Identifying numerical and categorical features
    - 3.7 Preprocessing and encoding
    - 3.8 Building the preprocessing pipeline
    - 3.9 Leakage considerations

    - 3.10 Baseline model
    - 3.11 Logistic Regression
    - 3.12 Decision Tree Classifier
    - 3.13 Random Forest Classifier

3. Model evaluation
4. Model comparison
5. Classification findings

6. Overall Modeling Conclusions

7. Limitations

8. Recommendations / Future Work

In [1]:
# specify libraries to use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling libraries
# Baseline model
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Split the data
from sklearn.model_selection import train_test_split, GridSearchCV

# Metrics
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay

# Decision Tree model
from sklearn.tree import DecisionTreeClassifier

# Random Forest model
from sklearn.ensemble import RandomForestClassifier

In [2]:
# load dataset
math_df = pd.read_csv("student-mat.csv", sep=";")
math_df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [3]:
math_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [4]:
# Open and read the txt file
with open('student.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Print the text to the VS Code terminal
print(content)


# Attributes for both student-mat.csv (Math course) and student-por.csv (Portuguese language course) datasets:
1 school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)
2 sex - student's sex (binary: "F" - female or "M" - male)
3 age - student's age (numeric: from 15 to 22)
4 address - student's home address type (binary: "U" - urban or "R" - rural)
5 famsize - family size (binary: "LE3" - less or equal to 3 or "GT3" - greater than 3)
6 Pstatus - parent's cohabitation status (binary: "T" - living together or "A" - apart)
7 Medu - mother's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
8 Fedu - father's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
9 Mjob - mother's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or 

## Creating the Pass/Fail Target

In [5]:
math_df['G3'].head(10)

0     6
1     6
2    10
3    15
4    10
5    15
6    11
7     6
8    19
9    15
Name: G3, dtype: int64

In [6]:
# Create binary pass/fail target
math_df['pass_fail'] = (math_df['G3'] >= 10).astype(int)

# Display the first few observations
math_df[['G3', 'pass_fail']].head()

,G3,pass_fail
0,6,0
1,6,0
2,10,1
3,15,1
4,10,1


In [10]:
# verify the transformation
# Check pass/fail counts
print(math_df['pass_fail'].value_counts())

# Check pass/fail proportions
print(math_df['pass_fail'].value_counts(normalize=True).round(3)*100)

pass_fail
1    265
0    130
Name: count, dtype: int64
pass_fail
1    67.1
0    32.9
Name: proportion, dtype: float64


**Interpretation:**

### 3.3 Class Distribution

The resulting target contains 265 students classified as passing and
130 students classified as failing. This corresponds to a class
distribution of 67.1% Pass and 32.9% Fail.

The classes are moderately imbalanced, with Pass representing the
majority class. The imbalance is not severe, but it means that accuracy
alone may not provide a complete assessment of classification performance.

A baseline classifier will therefore be established before training the
more complex models. In particular, its performance will provide a
reference point for determining whether the trained models offer
meaningful predictive improvement over the majority-class distribution.